In [11]:
# =========================
# 🔹 INSTALL LIBRARIES
# =========================
!pip install vncorenlp transformers torch scikit-learn tqdm

# =========================
# 🔹 DOWNLOAD VnCoreNLP
# =========================
import os

vncorenlp_dir = "VnCoreNLP-master"
vncorenlp_zip = "master.zip"

if not os.path.exists(vncorenlp_dir):
    print(f"Downloading {vncorenlp_zip}...")
    !wget -q https://github.com/vncorenlp/VnCoreNLP/archive/refs/heads/{vncorenlp_zip}
    print(f"Unzipping {vncorenlp_zip}...")
    !unzip -q {vncorenlp_zip}
    print("VnCoreNLP download and extraction complete.")
else:
    print("VnCoreNLP already exists. Skipping download and extraction.")

# =========================
# 🔹 DOWNLOAD DATASET (VNTC)
# =========================
vntc_dir = "VNTC"
if not os.path.exists(vntc_dir):
    print(f"Cloning {vntc_dir} repository...")
    !git clone https://github.com/duyvuleo/VNTC.git
    print("VNTC repository cloned.")
else:
    print("VNTC repository already exists. Skipping cloning.")

# =========================
# 🔹 UNRAR DATASET
# =========================
# Install unrar if not already present
!apt-get update -qq > /dev/null
!apt-get install -y unrar > /dev/null
print("Unrar installed.")

train_full_dir = "/content/VNTC/Data/10Topics/Ver1.1/Train_Full"
test_full_dir = "/content/VNTC/Data/10Topics/Ver1.1/Test_Full"

if not os.path.exists(train_full_dir) or not os.path.exists(test_full_dir):
    print("Extracting dataset archives...")
    !unrar x /content/VNTC/Data/10Topics/Ver1.1/Train_Full.rar /content/VNTC/Data/10Topics/Ver1.1/
    !unrar x /content/VNTC/Data/10Topics/Ver1.1/Test_Full.rar /content/VNTC/Data/10Topics/Ver1.1/
    print("Dataset extraction complete.")
else:
    print("Dataset already extracted. Skipping extraction.")

VnCoreNLP already exists. Skipping download and extraction.
VNTC repository already exists. Skipping cloning.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Unrar installed.
Dataset already extracted. Skipping extraction.


In [12]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from vncorenlp import VnCoreNLP
import random
import numpy as np
from collections import Counter
import time
# =========================
# 🔹 CONFIG
# =========================
TRAIN_DIR = "/content/VNTC/Data/10Topics/Ver1.1/Train_Full"
TEST_DIR  = "/content/VNTC/Data/10Topics/Ver1.1/Test_Full"

BATCH_SIZE = 16
EPOCHS = 5
MAX_LENGTH = 128
MAX_PER_CLASS = 700

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# 🔹 SEED
# =========================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# =========================
# 🔹 VnCoreNLP
# =========================
jar_path = "VnCoreNLP-master/VnCoreNLP-1.1.1.jar"

rdrsegmenter = None

def get_segmenter():
    global rdrsegmenter
    if rdrsegmenter is None:
        rdrsegmenter = VnCoreNLP(jar_path, annotators="wseg", max_heap_size='-Xmx2g')
    return rdrsegmenter

def segment_text(text):
    try:
        text = text[:1000]
        seg = get_segmenter()
        sents = seg.tokenize(text)
        return " ".join([" ".join(s) for s in sents])
    except:
        return ""

def compute_class_weights(labels, num_classes):
    counter = Counter(labels)
    total = sum(counter.values())
    weights = []

    for i in range(num_classes):
        # weight = total / (num_classes * count_i)
        weights.append(total / (num_classes * counter[i]))

    return torch.tensor(weights, dtype=torch.float).to(device)
# =========================
# 🔹 LOAD DATA
# =========================
def read_file_safe(path):
    for enc in ["utf-8", "utf-16", "latin-1"]:
        try:
            with open(path, encoding=enc) as f:
                text = f.read().strip()

                # Skip only truly bad text
                if len(text) < 20:
                    return None

                return text
        except:
            continue
    return None

def load_data(data_dir, label_map=None, max_per_class=700):
    texts, labels = [], []

    if label_map is None:
        label_map = {}

    if not os.path.exists(data_dir):
        print(f"Warning: Data directory not found: {data_dir}")
        return texts, labels, label_map

    print(f"Loading data from: {data_dir}")

    for label in sorted(os.listdir(data_dir)):
        if label not in label_map:
            label_map[label] = len(label_map)

        folder = os.path.join(data_dir, label)
        if not os.path.isdir(folder):
            continue

        files_in_folder = os.listdir(folder)

        count = 0

        for file in files_in_folder:
            if max_per_class is not None and count >= max_per_class:
                break

            text = read_file_safe(os.path.join(folder, file))
            if not text:
                continue

            texts.append(text)
            labels.append(label_map[label])
            count += 1

    print(f"Loaded {len(texts)} samples from {data_dir}")
    return texts, labels, label_map

# =========================
# 🔹 VOCAB
# =========================
def build_vocab(texts, max_vocab_size=20000):
    counter = Counter()

    for text in texts:
        counter.update(text.lower().split())

    vocab = {"<PAD>": 0, "<UNK>": 1}

    for i, (word, _) in enumerate(counter.most_common(max_vocab_size)):
        vocab[word] = i + 2

    return vocab

def text_to_seq(text, vocab):
    tokens = text.lower().split()
    seq = [vocab.get(t, vocab["<UNK>"]) for t in tokens]

    if len(seq) < MAX_LENGTH:
        seq += [vocab["<PAD>"]] * (MAX_LENGTH - len(seq))
    else:
        seq = seq[:MAX_LENGTH]

    return seq

# =========================
# 🔹 DATASET
# =========================
class LSTMDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(text_to_seq(self.texts[idx], self.vocab)),
            "labels": torch.tensor(self.labels[idx])
        }

# =========================
# 🔹 DATALOADER
# =========================
from sklearn.model_selection import train_test_split

def build_dataloaders():
    train_texts, train_labels, label_map = load_data(TRAIN_DIR)
    test_texts, test_labels, _ = load_data(TEST_DIR, label_map)

    if len(train_texts) == 0:
        print("Error: No training data loaded. Cannot perform train_test_split.")
        raise ValueError("No training data available.")

    train_texts, val_texts, train_labels, val_labels = train_test_split(
        train_texts, train_labels,
        test_size=0.2,
        stratify=train_labels,
        random_state=42
    )

    print("Segmenting text...")
    train_texts = [segment_text(t) for t in tqdm(train_texts)]
    val_texts   = [segment_text(t) for t in tqdm(val_texts)]
    test_texts  = [segment_text(t) for t in tqdm(test_texts)]

    filtered_train_data = [(t, l) for t, l in zip(train_texts, train_labels) if t]
    train_texts, train_labels = zip(*filtered_train_data) if filtered_train_data else ([], [])

    filtered_val_data = [(t, l) for t, l in zip(val_texts, val_labels) if t]
    val_texts, val_labels = zip(*filtered_val_data) if filtered_val_data else ([], [])

    filtered_test_data = [(t, l) for t, l in zip(test_texts, test_labels) if t]
    test_texts, test_labels = zip(*filtered_test_data) if filtered_test_data else ([], [])

    if len(train_texts) == 0:
        print("Error: No training data left after text segmentation. Cannot build vocab or dataloaders.")
        raise ValueError("No training data available after segmentation.")

    vocab = build_vocab(train_texts)

    train_ds = LSTMDataset(train_texts, train_labels, vocab)
    val_ds   = LSTMDataset(val_texts, val_labels, vocab)
    test_ds  = LSTMDataset(test_texts, test_labels, vocab)

    return (
        DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True),
        DataLoader(val_ds, batch_size=BATCH_SIZE),
        DataLoader(test_ds, batch_size=BATCH_SIZE),
        len(label_map),
        len(vocab),
        test_texts,
        label_map
    )

# =========================
# 🔹 MODEL
# =========================
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, 300)
        self.lstm = nn.LSTM(300, 256, batch_first=True)
        self.fc = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.embedding(x)
        _, (h, _) = self.lstm(x)
        x = self.dropout(h[-1])
        return self.fc(x)

# =========================
# 🔹 TRAIN / EVAL
# =========================
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for batch in tqdm(loader):
        optimizer.zero_grad()

        x = batch["input_ids"].to(device)
        y = batch["labels"].to(device)

        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

def evaluate(model, loader, texts=None, label_map=None, print_per_class=False, show_errors=True):
    model.eval()
    preds, labels = [], []

    with torch.no_grad():
        for batch in loader:
            x = batch["input_ids"].to(device)
            y = batch["labels"].to(device)

            logits = model(x)
            pred = torch.argmax(logits, 1)

            preds.extend(pred.cpu().numpy())
            labels.extend(y.cpu().numpy())

    preds = np.array(preds)
    labels = np.array(labels)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")

    print(f"\n✅ Accuracy: {acc:.4f} | F1: {f1:.4f}")
    # =========================
    # 🔥 Most Confused Pairs
    # =========================
    if show_errors:
        from collections import Counter

        pairs = Counter()

        for t, p in zip(labels, preds):
            if t != p:
                pairs[(t, p)] += 1

        print("\n🔥 Most confused pairs:")
        inv_map = None
        if label_map:
            inv_map = {v: k for k, v in label_map.items()}

        for (t, p), count in pairs.most_common(5):
            if inv_map:
                t = inv_map[t]
                p = inv_map[p]
            print(f"{t} → {p}: {count}")
    # =========================
    # 📊 Per-class metrics
    # =========================
    if print_per_class:
        report = classification_report(labels, preds, digits=4)
        print("\n📊 Per-Class Metrics:\n")
        print(report)

    # =========================
    # 🔥 Confusion Matrix
    # =========================
    if show_errors:
        cm = confusion_matrix(labels, preds)
        print("\n🔥 Confusion Matrix:\n")
        print(cm)

    # =========================
    # ❌ Error Analysis
    # =========================
    if show_errors and texts is not None:
        print("\n❌ Misclassified Examples:\n")

        mis_idx = np.where(preds != labels)[0]

        # show 5 examples
        for i in mis_idx[:5]:
            true_label = labels[i]
            pred_label = preds[i]

            if label_map:
                inv_map = {v: k for k, v in label_map.items()}
                true_label = inv_map[true_label]
                pred_label = inv_map[pred_label]

            print(f"TEXT: {texts[i][:200]}...")
            print(f"TRUE: {true_label} | PRED: {pred_label}")
            print("-" * 60)

    return acc, f1

# =========================
# 🔹 MAIN
# =========================
def main():
    print("Using device:", device)

    try:
        train_loader, val_loader, test_loader, num_classes, vocab_size, test_texts, label_map = build_dataloaders()
    except ValueError as e:
        print(f"Initialization error: {e}")
        return # Exit main if data loading fails

    model = LSTMClassifier(vocab_size, num_classes).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    # 🔹 Compute class weights
    all_train_labels = []
    for batch in train_loader:
        all_train_labels.extend(batch["labels"].numpy())
    class_weights = compute_class_weights(all_train_labels, num_classes)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    train_time = 0
    for epoch in range(EPOCHS):
        print(f"\nEpoch {epoch+1}/{EPOCHS}")

        if len(train_loader) == 0:
            print("Skipping training due to empty train_loader.")
            break
        start_epoch = time.time()
        loss = train_epoch(model, train_loader, optimizer, criterion)
        train_time += time.time() - start_epoch
        if len(val_loader) == 0:
            print("Skipping validation due to empty val_loader.")
            acc, f1 = 0.0, 0.0
        else:
            acc, f1 = evaluate(
                model,
                val_loader,
                print_per_class=False,
                show_errors=False
            )

        print(f"Loss: {loss:.4f}")
        print(f"Val Acc: {acc:.4f} | F1: {f1:.4f}")
        print(f"Training time: {train_time:.2f}s")
    print("\nTesting...")
    start_test = time.time()
    if len(test_loader) == 0:
        print("Skipping testing due to empty test_loader.")
        acc, f1 = 0.0, 0.0
    else:
        acc, f1 = evaluate(
            model,
            test_loader,
            print_per_class=False,
            show_errors=False
        )
    print(f"Test Acc: {acc:.4f} | F1: {f1:.4f}")
    print(f"Inference time: {time.time() - start_test:.2f}s")
    # =====================
    # ERROR ANALYSIS (not timed)
    # =====================
    evaluate(
        model,
        test_loader,
        texts=test_texts,
        label_map=label_map,
        print_per_class=True,
        show_errors=True
    )
main()

Using device: cpu
Loading data from: /content/VNTC/Data/10Topics/Ver1.1/Train_Full
Loaded 7000 samples from /content/VNTC/Data/10Topics/Ver1.1/Train_Full
Loading data from: /content/VNTC/Data/10Topics/Ver1.1/Test_Full
Loaded 7000 samples from /content/VNTC/Data/10Topics/Ver1.1/Test_Full
Segmenting text...


100%|██████████| 7000/7000 [01:09<00:00, 100.98it/s]



Epoch 1/5


100%|██████████| 350/350 [01:47<00:00,  3.26it/s]



✅ Accuracy: 0.2436 | F1: 0.2105
Loss: 2.1837
Val Acc: 0.2436 | F1: 0.2105
Training time: 107.26s

Epoch 2/5


...


✅ Accuracy: 0.5707 | F1: 0.5694
Loss: 1.2184
Val Acc: 0.5707 | F1: 0.5694
Training time: 279.14s

Epoch 4/5


100%|██████████| 350/350 [01:24<00:00,  4.15it/s]



✅ Accuracy: 0.6321 | F1: 0.6274
Loss: 0.7988
Val Acc: 0.6321 | F1: 0.6274
Training time: 363.44s

Epoch 5/5


100%|██████████| 350/350 [01:24<00:00,  4.12it/s]



✅ Accuracy: 0.6800 | F1: 0.6796
Loss: 0.4571
Val Acc: 0.6800 | F1: 0.6796
Training time: 448.43s

Testing...

✅ Accuracy: 0.6429 | F1: 0.6354
Test Acc: 0.6429 | F1: 0.6354
Inference time: 24.24s

✅ Accuracy: 0.6429 | F1: 0.6354

🔥 Most confused pairs:
Chinh tri Xa hoi → Kinh doanh: 222
Doi song → Khoa hoc: 146
Doi song → Van hoa: 143
Vi tinh → Khoa hoc: 102
Khoa hoc → The gioi: 90

📊 Per-Class Metrics:

              precision    recall  f1-score   support

           0     0.4519    0.2614    0.3312       700
           1     0.5704    0.3414    0.4272       700
           2     0.4513    0.6229    0.5234       700
           3     0.5825    0.8471    0.6903       700
           4     0.8111    0.6071    0.6944       700
           5     0.8406    0.6857    0.7553       700
           6     0.5750    0.8271    0.6784       700
           7     0.9431    0.8057    0.8690       700
           8     0.6645    0.7329    0.6970       700
           9     0.6778    0.6971    0.6873       7